# Worked Example: Catching an Overfit Strategy with the VALID Checklist

Two deliberately generic strategies on deterministic synthetic data:

- **MinedRSI** — 360 fast RSI configs; the "best" one is picked by **in-sample gross Sharpe** (two classic mistakes at once: selecting on IS performance, and ignoring costs at selection time).
- **HonestSMA** — a pre-committed SMA 50/200 crossover with an honest 9-config neighborhood.

Both go through the full 12-item VALID checklist (CPCV+PBO, Var(SR_IS), permutation test, cost sensitivity, baselines, ...) plus a Romano-Wolf / Deflated-Sharpe epilogue. Every number is computed live — run the notebook and watch the mined strategy fail.

> Run from the repo root (`jupyter lab` after `pip install -e ".[dev]"`). Runtime ≈ 1 min.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from examples.worked_example import main

## Run the full example

`main()` mines the RSI grid, evaluates both strategies through `valid.checklist.VALIDChecker.run_all`, runs the multiple-testing epilogue, and writes reports + figures to `results/worked_example/`.

In [ ]:
summary = main(data="synthetic", smoke=False)
summary

## The money shot: in-sample rank does not survive out-of-sample

The scatter shows every mined config's mean in-sample vs out-of-sample Sharpe across the 15 CPCV paths — the IS-selected "winner" (star) is nowhere special out-of-sample. The survivor chart shows what's left of the 360 configs after selection-aware corrections: nothing.

In [ ]:
from IPython.display import Image, display
out = Path("results/worked_example")
for fig in ["fig_is_oos_scatter.png", "fig_survivors.png", "fig_equity_curves.png"]:
    display(Image(filename=out / fig))

## Takeaways

1. **A shiny in-sample Sharpe is the cheapest thing in quant finance.** Mining 360 configs bought a big IS number that collapsed on the holdout.
2. **No single test is enough.** PBO alone was fooled here (the Witzany critique the paper confirms empirically) — the mined strategy was caught by the permutation test, the baseline comparison, and the Romano-Wolf/DSR epilogue.
3. **Honest and boring survives.** The pre-committed SMA scored higher on the checklist and kept its (modest) Sharpe out-of-sample.

Paper: [SSRN 6508779](https://ssrn.com/abstract=6508779) · Framework: [github.com/orcajae/valid-framework](https://github.com/orcajae/valid-framework)